# Datamine MSO2NPV Process Example

This notebook demonstrates how to configure and run the **`mso2npv`** process wrapper in `dmstudio`.

## Process Description

## MSO2NPV

Update a model containing ore and waste categories into a model suitable for strategic planning, using **[Mineable Shape Optimizer (MSO)](<../MSO/MSOv3_default.md>)** output wireframes.

[![](../Images/MSO2NPV.png)](<javascript:void\(0\);>)

A MODOUT1 model generated by MSO2NPV shown with a cutaway of stope wireframes.

The metal content and mass from the stope wireframe data is the same in the block model for the cells representing the MSO shapes, although an optional comparison table is available for analysis. All cells outside the MSO volumes are not considered ore. This process is intended to apply the selective mining units generated by MSO (or by any other means) into the model to be used for optimizing in **Studio NPVS**.

### Input Files:

* **modelin** (Block Model):
  Input model file containing ore and waste. This can be a normal or a rotated model
  Required=Yes

* **wiretr** (Wireframe Triangles):
  Input wireframe triangle file used to define the stopes. The file must also include the
  valuation fields (TONNES, VOLUME and grades) derived from MSO.
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe point file.
  Required=Yes

### Output Files:

* **modout1** (Block Model):
  Output model file containing just cells within the stope wireframes.
  Required=Yes

* **modout2** (Block Model):
  Output model model created by adding MODOUT1 to MODELIN
  Required=No

* **compare** (Table):
  Table showing the difference in volume and density, for each stope, between the input
  MSO values in the wireframe triangle file and the output MODOUT1 model.
  Required=No

### Fields:

* **stopeid** (Any : WIRETR):
  Name of Stope Identifier field in the WIRETR file.
  Default=Undefined
  Required=Yes

* **volume** (Any : WIRETR):
  Name of field defining volume of stope in the input wireframe triangle file.
  Default=Undefined
  Required=Yes

* **tonnes** (Any : WIRETR):
  Name of field defining tonnage of stope in the input wireframe triangle file.
  Default=Undefined
  Required=Yes

* **f1_f20** (Any : WIRETR):
  Grade field to be copied from WIRETR to the output models. F1 is mandatory.
  Default=Undefined
  Required=No

* **instope** (Any : -):
  Name of field to be created in the output model files to identify cells that lie inside
  a stope: =0 cell not in stope, =1 cell in stope
  Default=Undefined
  Required=No

### Parameters:

* **splits**:
  Maximum amount of splitting to be allowed. =0 : no splitting: parent cell. =1 : 1

* **split** (2 x 2 subcells. =2 : 2 splits: 4 x 4 subcells. =3 : 3 splits: 8 x 8 subcells.):
  Range=0-3
  Values=0,1,2,3
  Default=3
  Required=No

* **xsubcell**:
  Number of subcells per parent cell in X direction. Max 16. Only used if SPLITS=0
  Range=1-16
  Values=1-16
  Default=1
  Required=No

* **ysubcell**:
  Number of subcells per parent cell in Y direction (1). Max 16. Only used if SPLITS=0
  Range=1-16
  Values=1-16
  Default=1
  Required=No

* **resol**:
  Defines boundary resolution in Z direction. =0 : precise boundary location. =N :
  boundary rounded to nearest 1/Nth of parent cell size.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('mso2npv')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute mso2npv
print("Running mso2npv...")
dm_cmd.mso2npv(
    modelin_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    modouts_o=['t_mso2npv_out'],  # required
    compare_o='t_mso2npv_out',  # required
    stopeid_f='optional',  # required
    volume_f='optional',  # required
    tonnes_f=['optional_field'],  # required
    # f1_f20_f='optional',  # optional
    # instope_f='optional',  # optional
    # splits_p=3,  # optional
    # xsubcell_p=1,  # optional
    # ysubcell_p=1,  # optional
    # resol_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("mso2npv execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_mso2npv_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")